# Telco Genie Learning Day — Metric views
Run **after** `02_uc_documentation_and_constraints.py`. Creates [Metric Views](https://docs.databricks.com/aws/en/metric-views/create) with **YAML 1.1**, dimensions, measures, **`comment`** on each dimension and measure, and **agent metadata** (`display_name`, `synonyms`, `format`) for Genie.

| View | Grain / use |
|------|-------------|
| `telco_monthly_revenue_and_usage` | Customer-month — ARPU, plan mix, regional usage |
| `telco_support_experience` | Ticket — NPS, resolution, CX by region |
| `telco_network_reliability` | Event — outages/maintenance by location |

**Requires:** SQL warehouse or cluster on **Databricks Runtime 17.3+** ([prerequisites](https://docs.databricks.com/aws/en/metric-views/create)).

## Configuration

In [0]:
# Match 01_generate_dataset.py / 02_uc_documentation_and_constraints.py
CATALOG = "workspace"
SCHEMA = "telco"

spark.sql(f"USE {CATALOG}.{SCHEMA}")
print(f"Target: {CATALOG}.{SCHEMA}")

## 1. `telco_monthly_revenue_and_usage`
Nested joins (`customers` → `plans`) require **snowflake path** column refs: `customers.plans.<col>`, not `plans.<col>`. See [YAML joins](https://docs.databricks.com/aws/en/business-semantics/metric-views/yaml-reference#snowflake-schema-joins).

In [0]:
MV_USAGE_YAML = f"""
version: 1.1
comment: |
  Telco Learning Day: monthly usage and revenue at customer-month grain.
  Joins usage to customers and plans for ARPU, regional, and plan-tier analysis.
source: {CATALOG}.{SCHEMA}.usage
joins:
  - name: customers
    source: {CATALOG}.{SCHEMA}.customers
    on: source.customer_id = customers.customer_id
    joins:
      - name: plans
        source: {CATALOG}.{SCHEMA}.plans
        on: customers.plan_id = plans.plan_id
dimensions:
  - name: bill_month
    expr: source.month
    display_name: Billing month
    comment: Calendar month for the usage row; aligns usage and revenue to a billing period.
    synonyms:
      - month
      - billing period
  - name: region_type
    expr: customers.region_type
    display_name: Metro or regional
    comment: Whether the subscriber is classified as metro or regional for coverage and pricing context.
    synonyms:
      - metro
      - regional
      - location type
  - name: subscriber_state
    expr: customers.state
    display_name: State
    comment: Australian state or territory on the customer record used for regional reporting.
  - name: plan_name
    expr: customers.plans.plan_name
    display_name: Plan
    comment: Commercial mobile plan name from the plan dimension joined via the customer.
    synonyms:
      - mobile plan
      - tier
  - name: plan_category
    expr: customers.plans.plan_category
    display_name: Prepaid or postpaid
    comment: Billing model for the plan (prepaid vs postpaid), useful for mix and ARPU comparisons.
  - name: subscriber_status
    expr: customers.status
    display_name: Active or churned
    comment: Subscription lifecycle status at the time of the join (e.g. active vs churned).
    synonyms:
      - churn
      - customer status
measures:
  - name: customer_months
    expr: COUNT(*)
    display_name: Customer-months
    comment: Number of customer-month rows at this grain; use as a denominator for rates when grouped.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: total_data_gb
    expr: SUM(source.data_used_gb)
    display_name: Total data used (GB)
    comment: Aggregate mobile data consumed in gigabytes across the selected dimensions.
    synonyms:
      - data volume
      - usage
  - name: total_monthly_plan_aud
    expr: SUM(customers.plans.monthly_cost_aud)
    display_name: Total monthly plan fees (AUD)
    comment: Sum of recurring monthly plan charges in AUD before overages.
    format:
      type: currency
      currency_code: AUD
  - name: total_overage_aud
    expr: SUM(source.overage_charges_aud)
    display_name: Total overage charges (AUD)
    comment: Sum of charges beyond the plan allowance in AUD for the same grain as usage.
    format:
      type: currency
      currency_code: AUD
  - name: arpu_aud
    expr: AVG(customers.plans.monthly_cost_aud + source.overage_charges_aud)
    display_name: ARPU (AUD)
    comment: Average monthly revenue per customer-month (plan fee plus overage).
    synonyms:
      - average revenue per user
      - ARPU
      - revenue per user
    format:
      type: currency
      currency_code: AUD
"""

spark.sql(
    f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.telco_monthly_revenue_and_usage
WITH METRICS
LANGUAGE YAML
AS $${MV_USAGE_YAML}$$
"""
)
print(f"Created {CATALOG}.{SCHEMA}.telco_monthly_revenue_and_usage")

## 2. `telco_support_experience`

In [0]:
MV_SUPPORT_YAML = f"""
version: 1.1
comment: |
  Support tickets with customer region for NPS, resolution time, and channel analysis.
source: {CATALOG}.{SCHEMA}.support_tickets
joins:
  - name: customers
    source: {CATALOG}.{SCHEMA}.customers
    on: source.customer_id = customers.customer_id
dimensions:
  - name: ticket_category
    expr: source.category
    display_name: Ticket category
    comment: High-level support issue type (e.g. billing, network, device) for volume and trend analysis.
    synonyms:
      - complaint type
      - issue category
  - name: severity
    expr: source.severity
    display_name: Severity
    comment: Ticket priority or urgency tier used for SLA and backlog views.
  - name: channel
    expr: source.channel
    display_name: Contact channel
    comment: How the customer reached support (e.g. phone, chat, app) for channel mix reporting.
  - name: region_type
    expr: customers.region_type
    display_name: Metro or regional
    comment: Subscriber metro vs regional classification for experience comparisons by geography type.
  - name: ticket_month
    expr: "DATE_TRUNC('month', source.created_date)"
    display_name: Ticket month
    comment: Month the ticket was created; truncates created_date to month for time-series grouping.
measures:
  - name: ticket_count
    expr: COUNT(*)
    display_name: Ticket count
    comment: Number of support tickets matching the current filters and group-by dimensions.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: avg_nps
    expr: AVG(source.nps_score)
    display_name: Average NPS
    comment: Average Net Promoter-style score (0-10) on surveyed tickets.
    synonyms:
      - NPS
      - satisfaction
  - name: avg_resolution_hours
    expr: AVG(source.resolution_time_hours)
    display_name: Avg resolution (hours)
    comment: Mean hours from ticket creation to resolution for the selected slice.
    synonyms:
      - resolution time
"""

spark.sql(
    f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.telco_support_experience
WITH METRICS
LANGUAGE YAML
AS $${MV_SUPPORT_YAML}$$
"""
)
print(f"Created {CATALOG}.{SCHEMA}.telco_support_experience")

## 3. `telco_network_reliability`

In [0]:
MV_NETWORK_YAML = f"""
version: 1.1
comment: |
  Network outages and maintenance events by metro vs regional and event type.
source: {CATALOG}.{SCHEMA}.network_events
dimensions:
  - name: region_type
    expr: region_type
    display_name: Metro or regional
    comment: Whether the network event is attributed to metro or regional infrastructure.
  - name: state
    expr: state
    display_name: State
    comment: State or territory where the event is reported for regional reliability views.
  - name: event_type
    expr: event_type
    display_name: Event type
    comment: Classification of the incident (e.g. outage vs planned maintenance) for root-cause style splits.
    synonyms:
      - outage
      - maintenance
measures:
  - name: event_count
    expr: COUNT(*)
    display_name: Event count
    comment: Count of network events in the selected time and dimension slice.
  - name: avg_duration_min
    expr: AVG(duration_min)
    display_name: Avg duration (minutes)
    comment: Mean event duration in minutes across matching rows.
  - name: total_affected_customers
    expr: SUM(affected_customers)
    display_name: Total affected customers (modelled)
    comment: Sum of modelled subscriber impact counts; use for exposure-style reporting alongside duration.
"""

spark.sql(
    f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.telco_network_reliability
WITH METRICS
LANGUAGE YAML
AS $${MV_NETWORK_YAML}$$
"""
)
print(f"Created {CATALOG}.{SCHEMA}.telco_network_reliability")

## 4. Smoke query (optional)
Example: aggregate ARPU by plan from the usage metric view.

In [0]:
%sql
SELECT plan_name, MEASURE(arpu_aud) AS arpu_aud
FROM workspace.telco.telco_monthly_revenue_and_usage
GROUP BY plan_name
ORDER BY plan_name

## Done
**Next:** Configure the demo Genie space in the UI — **`04_genie_space_manual.md`** (overview + import: **`04_genie_space.md`**).